In [0]:
from pyspark import pipelines as dp
from pyspark.sql import functions as F
from pyspark.sql.window import Window

@dp.materialized_view(
    name="datawarehouse_project.gold.dim_customers",
    comment="Customer dimension with enriched data from multiple sources"
)
def dim_customers():
    # Read from silver tables in the pipeline
    cust_info = spark.read.table("silver.cust_info")
    cust_az12 = spark.read.table("Silver.cust_az12")
    loc_a101 = spark.read.table("Silver.loc_a101")
    
    # Join customer info with location and additional customer data
    df = cust_info.alias("ci") \
        .join(cust_az12.alias("ca"), F.col("ci.cst_key") == F.col("ca.cid"), "left") \
        .join(loc_a101.alias("la"), F.col("ci.cst_key") == F.col("la.cid"), "left")
    
    # Create customer dimension with surrogate key
    window_spec = Window.orderBy("ci.cst_id")
    
    return df.select(
        F.row_number().over(window_spec).alias("customer_key"),
        F.col("ci.cst_id").alias("customer_id"),
        F.col("ci.cst_key").alias("customer_num"),
        F.col("ci.cst_firstname").alias("first_name"),
        F.col("ci.cst_lastname").alias("last_name"),
        F.col("la.cntry").alias("country"),
        F.col("ci.cst_marital_status").alias("marital_status"),
        F.when(F.col("ci.cst_gndr") != "n/a", F.col("ci.cst_gndr"))
         .otherwise(F.coalesce(F.col("ca.gen"), F.lit("n/a"))).alias("gender"),
        F.col("ca.bdate").alias("birthday"),
        F.col("ci.cst_create_date").alias("create_date")
    )

In [0]:
@dp.materialized_view(
    name="datawarehouse_project.gold.dim_products",
    comment="Product dimension with category information and deduplication"
)
def dim_products():
    # Read from silver tables in the pipeline
    prd_info = spark.read.table("Silver.prd_info")
    px_cat = spark.read.table("Silver.px_cat_g_1_v_2")
    
    # Join products with categories
    df = prd_info.alias("pn") \
        .join(px_cat.alias("pc"), F.col("pn.cate_id") == F.col("pc.id"), "left")
    
    # Deduplicate products - keep most recent per product key
    window_dedup = Window.partitionBy("pn.prd_key").orderBy(
        F.col("pn.prd_start_dt").desc(),
        F.col("pn.prd_id").desc()
    )
    
    df_ranked = df.withColumn("rn", F.row_number().over(window_dedup)) \
        .filter(F.col("rn") == 1)
    
    # Create product dimension with surrogate key
    window_key = Window.orderBy("pn.prd_start_dt", "pn.prd_key")
    
    return df_ranked.select(
        F.row_number().over(window_key).alias("product_key"),
        F.col("pn.prd_id").alias("product_id"),
        F.col("pn.prd_key").alias("product_number"),
        F.col("pn.prd_nm").alias("product_name"),
        F.col("pn.cate_id").alias("category_id"),
        F.col("pc.cat").alias("category"),
        F.col("pc.subcat").alias("sub_category"),
        F.col("pc.maintenance"),
        F.col("pn.prd_cost").alias("cost"),
        F.col("pn.prd_line").alias("product_line"),
        F.col("pn.prd_start_dt").alias("start_date")
    )

In [0]:
@dp.materialized_view(
    name="fact_sales",
    comment="Sales fact table with foreign keys to customer and product dimensions"
)
def fact_sales():
    # Read from silver and gold tables in the pipeline
    sales_details = spark.read.table("Silver.sales_details")
    dim_products = spark.read.table("gold.dim_products")
    dim_customers = spark.read.table("gold.dim_customers")
    
    # Join sales with dimension tables to get surrogate keys
    df = sales_details.alias("sd") \
        .join(dim_products.alias("pr"), F.col("sd.sls_prd_key") == F.col("pr.product_number"), "left") \
        .join(dim_customers.alias("cu"), F.col("sd.sls_cust_id") == F.col("cu.customer_id"), "left")
    
    return df.select(
        F.col("sd.sls_ord_num").alias("order_number"),
        F.col("pr.product_key"),
        F.col("cu.customer_key"),
        F.col("sd.sls_sales").alias("sales_amount"),
        F.col("sd.sls_quantity").alias("quantity"),
        F.col("sd.sls_price").alias("price"),
        F.col("sd.sls_order_dt").alias("order_date"),
        F.col("sd.sls_ship_dt").alias("shipping_date"),
        F.col("sd.sls_due_dt").alias("due_date")
    )